In [2]:
import ipywidgets as widgets
from pykiba import serial_ports_list, PykiDev

availble_serial_ports = serial_ports_list()

port_selection =  widgets.Dropdown(
    options=[x.device for x in availble_serial_ports],
    value=None,
    description='Serial Port:',
    disabled=False,
)
display(port_selection)
port_baudrate = widgets.Dropdown(
    options=['9600', '115200'],
    value='115200',
    description='Baudrate:',
    disabled=False,
)
display(port_baudrate)

open_button = widgets.ToggleButton(
    value=False,
    description='Closed',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Port State',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)


ard = None
def on_open_button_click(b):
    global ard
    open_button.description='Closed'
    open_button.button_style=''
    if ard:
        ard.close()
        ard = None
        
    if open_button.value:
        if (ard is None) and (port_selection.value is not None):
            open_button.description='....'
            open_button.button_style='warning'
            ard = PykiDev(port =  port_selection.value, baudrate = int(port_baudrate.value))
            open_button.description='Open'
            open_button.button_style='success'
        else:  open_button.value = False

open_button.observe(on_open_button_click, 'value') 
display(open_button)

0.   None   /dev/ttyS0
1.   Arduino (www.arduino.cc)   /dev/ttyACM0


Dropdown(description='Serial Port:', options=('/dev/ttyS0', '/dev/ttyACM0'), value=None)

Dropdown(description='Baudrate:', index=1, options=('9600', '115200'), value='115200')

ToggleButton(value=False, description='Closed', icon='check', tooltip='Port State')

In [52]:
registers = ard.globs()

if any(filter(lambda x: type(x) == float, registers[0])):
    for reg in registers:
        if(reg[3] - reg[2]) > 0.5:
            reg_min = reg[2]
            reg_max = reg[3]
        else:
            reg_min = -1000000
            reg_max = 1000000
            
        sl = widgets.FloatSlider(
                 value=reg[1],
                 min=reg_min,
                 max=reg_max,
                 step=(reg[3]-reg[2])/1000,
                 description=reg[0],
                 disabled=False,
                 continuous_update=True,
                 orientation='horizontal',
                 readout=True,
                 readout_format='.3f'
                 )
            
        bt = widgets.Button(
                 description=reg[0],
                 disabled=False,
                 button_style='', # 'success', 'info', 'warning', 'danger' or ''
                 tooltip='',
                 icon='check' # (FontAwesome names without the `fa-` prefix
                 )
        
        def b_function(b, slider=sl, button=bt):
             slider.value = ard.command(button.description, slider.value)
           
            
        bt.on_click(b_function)
        display(widgets.HBox([sl,bt]))
        
        
        

In [45]:
 reg = registers[4]

sl = widgets.FloatSlider(value = reg[1] , min = reg[2], max = reg[3] ,continuous_update=True) 
btn = widgets.Button(description= reg[0])
def oncl(b):
    sl.value = ard.command(reg[0], sl.value)
btn.on_click(oncl)
display(widgets.HBox([sl,btn]))



In [51]:
ard.globs()

[['w1', 1.2, 0.0, 50.0, 1],
 ['w2', 4.85, 0.0, 50.0, 1],
 ['a1', 2.07, 0.01, 10.0, 1],
 ['a2', 4.61, 0.01, 10.0, 1],
 ['f1', 2.24, 0.0, 0.0, 1],
 ['f2', 13.57, 0.0, 0.0, 1],
 ['t', 768.0, 0.0, 0.0, 1]]

In [39]:
ard.command('f1', 1.2)

8.68

In [10]:
bnfname = f'fn{registers[0][0]}' 
print(create_function_from_string(bnfname, "return ard.command(registers[0][0])"))   

NameError: name 'create_function_from_string' is not defined

In [8]:
ard.command(registers[0][0])

0.01

In [9]:
print(registers)

[['w1', 0.01, 0.0, 50.0, 1], ['w2', 0.03, 0.0, 50.0, 1], ['a1', 2.0, 0.01, 10.0, 1], ['a2', 3.0, 0.01, 10.0, 1], ['f1', 5.9, 0.0, 0.0, 1], ['f2', 4.29, 0.0, 0.0, 1], ['t', 684.0, 0.0, 0.0, 1]]
